DSC550 - Week 3
Kathryn Hess
6/22/2025

Download the labeled training dataset from this link: Bag of Words Meets Bags of Popcorn. 
Part 1: Using the TextBlob Sentiment Analyzer
1.	Import the movie review data as a data frame and ensure that the data is loaded properly.


In [3]:
import pandas as pd

#import the CSV
df = pd.read_csv("C:\\Users\\hessk\\OneDrive\\Desktop\\DSC550\\Movie Reviews.csv")

# Display the first few rows to ensure it's loaded properly
print(df.head())
print(df.info())

       id  sentiment                                             review
0  5814_8          1  With all this stuff going down at the moment w...
1  2381_9          1  \The Classic War of the Worlds\" by Timothy Hi...
2  7759_3          0  The film starts with a manager (Nicholas Bell)...
3  3630_4          0  It must be assumed that those who praised this...
4  9495_8          1  Superbly trashy and wondrously unpretentious 8...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         25000 non-null  object
 1   sentiment  25000 non-null  int64 
 2   review     25000 non-null  object
dtypes: int64(1), object(2)
memory usage: 586.1+ KB
None


2.	How many of each positive and negative reviews are there?

In [11]:
#count the values for different values, 0 and 1
print(df['sentiment'].value_counts())

sentiment
1    12500
0    12500
Name: count, dtype: int64


3.	Use TextBlob to classify each movie review as positive or negative. Assume that a polarity score greater than or equal to zero is a positive sentiment and less than 0 is a negative sentiment.

In [24]:
import pandas as pd
from textblob import TextBlob

def classify_sentiment(text):
    polarity = TextBlob(str(text)).sentiment.polarity
    return 'positive' if polarity >= 0 else 'negative'

df['sentiment'] = df['review'].apply(classify_sentiment)

print(df[['review', 'sentiment']].head())

                                              review sentiment
0  With all this stuff going down at the moment w...  positive
1  \The Classic War of the Worlds\" by Timothy Hi...  positive
2  The film starts with a manager (Nicholas Bell)...  negative
3  It must be assumed that those who praised this...  positive
4  Superbly trashy and wondrously unpretentious 8...  negative


In [33]:
#count positive/negative reviews
print(df['sentiment'].value_counts())

sentiment
positive    19017
negative     5983
Name: count, dtype: int64


In [53]:
df['truesentiment'] = df['review'].apply(classify_sentiment)

#create a column to represent the postive/negative value
print(df[['review', 'truesentiment']].head())


                                              review truesentiment
0  With all this stuff going down at the moment w...      positive
1  \The Classic War of the Worlds\" by Timothy Hi...      positive
2  The film starts with a manager (Nicholas Bell)...      negative
3  It must be assumed that those who praised this...      positive
4  Superbly trashy and wondrously unpretentious 8...      negative


4.	Check the accuracy of this model. Is this model better than random guessing?

In [51]:
print(df.columns)

Index(['id', 'sentiment', 'review', 'truesentiment'], dtype='object')


In [55]:
accuracy = (df['sentiment'] == df['truesentiment']).mean()
print(f"Model accuracy: {accuracy:.2%}")

Model accuracy: 100.00%


The model is better than random guessing - it is right 100% of the time. 

5.	For up to five points extra credit, use another prebuilt text sentiment analyzer, e.g., VADER, and repeat steps (3) and (4).

In [66]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    score = analyzer.polarity_scores(str(text))['compound']
    return 'positive' if score >= 0 else 'negative'

df['vader_sentiment'] = df['review'].apply(vader_sentiment)

In [68]:
accuracy = (df['vader_sentiment'] == df['truesentiment']).mean()
print(f"VADER accuracy: {accuracy:.2%}")

VADER accuracy: 78.82%


Part 2: Prepping Text for a Custom Model
If you want to run your own model to classify text, it needs to be in proper form to do so. The following steps will outline a procedure to do this on the movie reviews text.
1.	Convert all text to lowercase letters.


In [71]:
df['review'] = df['review'].str.lower()

2.	Remove punctuation and special characters from the text.

In [76]:
import re

df['review'] = df['review'].apply(lambda x: re.sub(r'[^a-z0-9\s]', '', str(x)))

print(df.head)

<bound method NDFrame.head of             id sentiment                                             review  \
0       5814_8  positive  with all this stuff going down at the moment w...   
1       2381_9  positive  the classic war of the worlds by timothy hines...   
2       7759_3  negative  the film starts with a manager nicholas bell g...   
3       3630_4  positive  it must be assumed that those who praised this...   
4       9495_8  negative  superbly trashy and wondrously unpretentious 8...   
...        ...       ...                                                ...   
24995   3453_3  positive  it seems like more consideration has gone into...   
24996   5064_1  positive  i dont believe they made this film completely ...   
24997  10905_3  positive  guy is a loser cant get girls needs to build u...   
24998  10194_3  positive  this 30 minute documentary buuel made in the e...   
24999   8478_8  positive  i saw this movie as a child and it broke my he...   

      truesentiment v

3.	Remove stop words.

In [84]:
import nltk
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = str(text).split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)

#remove stop words from column
df['review'] = df['review'].apply(remove_stopwords)

print(df.head)

<bound method NDFrame.head of             id sentiment                                             review  \
0       5814_8  positive  stuff going moment mj ive started listening mu...   
1       2381_9  positive  classic war worlds timothy hines entertaining ...   
2       7759_3  negative  film starts manager nicholas bell giving welco...   
3       3630_4  positive  must assumed praised film greatest filmed oper...   
4       9495_8  negative  superbly trashy wondrously unpretentious 80s e...   
...        ...       ...                                                ...   
24995   3453_3  positive  seems like consideration gone imdb reviews fil...   
24996   5064_1  positive  dont believe made film completely unnecessary ...   
24997  10905_3  positive  guy loser cant get girls needs build picked st...   
24998  10194_3  positive  30 minute documentary buuel made early 1930s o...   
24999   8478_8  positive  saw movie child broke heart story unfinished e...   

      truesentiment v

4.	Apply NLTK’s PorterStemmer.

In [89]:
import nltk
from nltk.stem import PorterStemmer

ps = PorterStemmer()

def stem_text(text):
    tokens = str(text).split()
    stemmed = [ps.stem(word) for word in tokens]
    return ' '.join(stemmed)

df['review'] = df['review'].apply(stem_text)

print(df.head)

<bound method NDFrame.head of             id sentiment                                             review  \
0       5814_8  positive  stuff go moment mj ive start listen music watc...   
1       2381_9  positive  classic war world timothi hine entertain film ...   
2       7759_3  negative  film start manag nichola bell give welcom inve...   
3       3630_4  positive  must assum prai film greatest film opera ever ...   
4       9495_8  negative  superbl trashi wondrou unpretenti 80 exploit h...   
...        ...       ...                                                ...   
24995   3453_3  positive  seem like consid gone imdb review film went so...   
24996   5064_1  positive  dont believ made film complet unnecessari firs...   
24997  10905_3  positive  guy loser cant get girl need build pick strong...   
24998  10194_3  positive  30 minut documentari buuel made earli 1930 one...   
24999   8478_8  positive  saw movi child broke heart stori unfinish end ...   

      truesentiment v

5.	Create a bag-of-words matrix from your stemmed text (output from (4)), where each row is a word-count vector for a single movie review (see sections 5.3 & 6.8 in the Machine Learning with Python Cookbook). Display the dimensions of your bag-of-words matrix. The number of rows in this matrix should be the same as the number of rows in your original data frame.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

#fit and transform to create a bag of words matrix
bow_matrix = vectorizer.fit_transform(df['review'])

#display the demtions
print("Bag-of-words shape:", bow_matrix.shape)
print("Original DataFrame:", df.shape[0])

Bag-of-words shape: (25000, 74849)
Original DataFrame: 25000


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Assume your stemmed text is in the 'Review' column
tfidf_vectorizer = TfidfVectorizer()

# Fit and transform the stemmed reviews to create the tf-idf matrix
tfidf_matrix = tfidf_vectorizer.fit_transform(df['review'])

# Display the dimensions of the matrix
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of rows in original DataFrame:", df.shape[0])